In [1]:
# ============================================================================
# IMPORTS
# ============================================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import wfdb
from tqdm.auto import tqdm

# Signal processing
from scipy.signal import butter, filtfilt, welch
from scipy.stats import skew, kurtosis
import neurokit2 as nk

# Bayesian modeling
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import multiprocessing

# Evaluation
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    adjusted_mutual_info_score, fowlkes_mallows_score,
    confusion_matrix, accuracy_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Jupyter-specific settings
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful")

✓ All imports successful


In [2]:
#import sys
#!{sys.executable} -m pip install seaborn

In [3]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Master configuration for the entire pipeline"""
    
    # Data paths
    PTBXL_ROOT = "/Users/reubenaddison/Downloads/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1/"
    
    # Feature extraction
    SAMPLE_RATE = 500  # Hz for _hr files
    WINDOW_SEC = 1     # 1-second windows
    TOTAL_SEC = 10     # 10-second records
    
    # Clustering
    N_CLUSTERS = 4
    N_DRAWS = 2000
    N_TUNE = 1000
    TARGET_ACCEPT = 0.95
    RANDOM_SEED = 42
    
    # CPU optimization
    total_cores = multiprocessing.cpu_count()
    N_CHAINS = 4
    N_CORES = max(1, total_cores - 2)  # Leave 2 for OS
    
    # Processing limits (set to None for full dataset)
    MAX_RECORDS = None  # None = all records, or set to 1000 for testing
    
    def __repr__(self):
        return f"Config(clusters={self.N_CLUSTERS}, cores={self.N_CORES}, max_records={self.MAX_RECORDS})"

config = Config()
print(config)


Config(clusters=4, cores=8, max_records=None)


In [4]:
# ============================================================================
# PART 1: DATA LOADING & PREPROCESSING
# ============================================================================

def load_ptbxl_database(root_path):
    """Load PTB-XL metadata"""
    db_path = os.path.join(root_path, "ptbxl_database.csv")
    db = pd.read_csv(db_path)
    print(f"✓ Loaded {len(db):,} records from PTB-XL database")
    return db


def create_degradation_labels(df):
    """
    Create binary labels: 0=clean, 1=degraded
    Based on noise columns in PTB-XL metadata
    """
    noise_cols = ["baseline_drift", "static_noise", "burst_noise", "electrodes_problems"]
    
    def to_flag(x):
        if pd.isna(x):
            return 0
        s = str(x).strip()
        if s == "" or s.lower() == "nan":
            return 0
        try:
            return int(float(s) != 0.0)
        except Exception:
            return 1
    
    noise_flags = df[noise_cols].applymap(to_flag)
    degraded = (noise_flags.sum(axis=1) > 0).astype(int).values
    
    print(f"✓ Created labels: {np.sum(degraded==0):,} clean, {np.sum(degraded==1):,} degraded")
    print(f"  Degradation rate: {degraded.mean():.1%}")
    
    return degraded

In [5]:
# ============================================================================
# PART 2: FEATURE EXTRACTION
# ============================================================================

def _trapz(y, x):
    """Robust trapezoidal integration"""
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    if hasattr(np, "trapz"):
        return float(np.trapz(y, x))
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    if y.size < 2:
        return 0.0
    dx = np.diff(x)
    return float(np.sum((y[:-1] + y[1:]) * 0.5 * dx))


def load_record_wfdb(root, rel_path):
    """Load ECG record from PTB-XL"""
    record_path = os.path.join(root, rel_path)
    sig, fields = wfdb.rdsamp(record_path)
    fs = float(fields["fs"])
    return sig, fs, fields


def welch_bandpowers(x, fs, nperseg=None):
    """
    Compute power in frequency bands:
    - p_base: 0-1 Hz (baseline drift)
    - p_qrs: 5-40 Hz (QRS complex)
    - p_hf: 40-100 Hz (high frequency noise)
    - p_tot: total power
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    
    if x.size < 16:
        return np.nan, np.nan, np.nan, np.nan
    
    if nperseg is None:
        nperseg = min(1024, x.size)
    if nperseg < 16:
        nperseg = min(256, x.size)
    
    f, pxx = welch(x, fs=fs, nperseg=nperseg, detrend="constant", scaling="density")
    
    nyq = fs / 2.0
    hi_tot = min(100.0, nyq)
    
    def bp(lo, hi):
        if hi <= lo:
            return 0.0
        idx = (f >= lo) & (f < hi)
        if not np.any(idx):
            return 0.0
        return _trapz(pxx[idx], f[idx])
    
    p_base = bp(0.0, 1.0)
    p_qrs = bp(5.0, 40.0)
    p_hf = bp(40.0, 100.0)
    p_tot = bp(0.0, hi_tot)
    
    return float(p_base), float(p_qrs), float(p_hf), float(p_tot)


def extract_record_features(sig_12lead, fs):
    """
    Extract spectral and statistical features from ECG
    Uses lead 0 as reference
    """
    sig_12lead = np.asarray(sig_12lead, dtype=float)
    
    if sig_12lead.ndim == 1:
        sig_12lead = sig_12lead[:, None]
    
    n_samples, n_ch = sig_12lead.shape
    
    feats = {
        "fs": float(fs),
        "n_samples": int(n_samples),
        "n_channels": int(n_ch)
    }
    
    # Use lead 0 (or could use lead II by selecting appropriate column)
    x = sig_12lead[:, 0]
    x = x[np.isfinite(x)]
    
    if x.size < 16:
        feats.update({
            "x_mean": np.nan, "x_std": np.nan, "x_rms": np.nan,
            "p_base": np.nan, "p_qrs": np.nan, "p_hf": np.nan, "p_tot": np.nan,
            "psqi": np.nan, "bassqi": np.nan, "hf_ratio": np.nan,
            "x_skew": np.nan, "x_kurtosis": np.nan
        })
        return feats
    
    # Time domain features
    feats["x_mean"] = float(np.mean(x))
    feats["x_std"] = float(np.std(x))
    feats["x_rms"] = float(np.sqrt(np.mean(x**2)))
    feats["x_skew"] = float(skew(x))
    feats["x_kurtosis"] = float(kurtosis(x, fisher=False))
    
    # Frequency domain features
    p_base, p_qrs, p_hf, p_tot = welch_bandpowers(x, fs)
    feats["p_base"] = p_base
    feats["p_qrs"] = p_qrs
    feats["p_hf"] = p_hf
    feats["p_tot"] = p_tot
    
    # Quality indices
    feats["psqi"] = float(p_qrs / p_tot) if np.isfinite(p_tot) and p_tot > 0 else np.nan
    feats["bassqi"] = float(p_base / p_tot) if np.isfinite(p_tot) and p_tot > 0 else np.nan
    feats["hf_ratio"] = float(p_hf / p_tot) if np.isfinite(p_tot) and p_tot > 0 else np.nan
    
    return feats


def build_feature_dataframe(db, y, root, n_records=None, use_hr=True):
    """
    Extract features from all records
    
    Parameters:
    -----------
    db : DataFrame with PTB-XL metadata
    y : binary labels
    root : PTB-XL root directory
    n_records : number of records to process (None = all)
    use_hr : use high-res (500Hz) or low-res (100Hz) files
    """
    if n_records is None:
        n_records = len(db)
    
    file_col = "filename_hr" if use_hr else "filename_lr"
    
    if file_col not in db.columns:
        raise KeyError(f"Column '{file_col}' not found in database")
    
    n_use = min(int(n_records), len(db))
    rows = []
    
    print(f"\n{'='*60}")
    print(f"Extracting features from {n_use:,} records...")
    print(f"{'='*60}")
    
    for i in tqdm(range(n_use), desc="Processing ECGs"):
        try:
            rel_path = db.loc[i, file_col]
            sig, fs, _ = load_record_wfdb(root, rel_path)
            
            feats = extract_record_features(sig, fs)
            feats["y"] = int(y[i])
            feats["ecg_id"] = int(db.loc[i, "ecg_id"]) if "ecg_id" in db.columns else int(i)
            feats["rel_path"] = rel_path
            
            rows.append(feats)
        except Exception as e:
            print(f"\n⚠️  Warning: Failed to process record {i}: {e}")
            continue
    
    feat_df = pd.DataFrame(rows)
    print(f"\n✓ Extracted features: shape {feat_df.shape}")
    print(f"  Features: {', '.join([c for c in feat_df.columns if c not in ['y', 'ecg_id', 'rel_path']])}")
    
    return feat_df



In [6]:
# ============================================================================
# PART 3: BAYESIAN CLUSTERING
# ============================================================================

def prepare_features_for_clustering(feat_df):
    """Standardize features and prepare for modeling"""
    # Drop non-feature columns
    drop_cols = ["y", "ecg_id", "rel_path"]
    X = feat_df.drop(columns=drop_cols, errors="ignore")
    y = feat_df["y"].values.astype(int)
    
    # Handle NaN
    X = X.fillna(0)
    X = np.asarray(X, dtype=np.float64)
    
    # Standardize
    X_mean = X.mean(axis=0)
    X_std = X.std(axis=0) + 1e-8
    X_standardized = (X - X_mean) / X_std
    
    y_prep = y.astype(np.float64)
    
    print(f"\n{'='*60}")
    print("Data prepared for clustering")
    print(f"{'='*60}")
    print(f"Samples: {X.shape[0]:,}")
    print(f"Features: {X.shape[1]}")
    print(f"Clean: {np.sum(y==0):,} | Degraded: {np.sum(y==1):,}")
    
    return X_standardized, y_prep, X_mean, X_std


def build_bayesian_model(X, y, K):
    """
    Build Student-T mixture model with Bernoulli observations
    
    Model structure:
    - Each cluster k has Student-T distributed features
    - Each cluster k has a degradation probability theta_k
    - Mixture weights pi determine cluster proportions
    """
    N, D = X.shape
    
    print(f"\n{'='*60}")
    print("Building Bayesian Mixture Model")
    print(f"{'='*60}")
    print(f"N = {N:,} samples")
    print(f"D = {D} features")
    print(f"K = {K} clusters")
    
    with pm.Model() as model:
        # Data containers
        X_data = pm.Data("X_data", X)
        y_data = pm.Data("y_data", y)
        
        # Priors: Cluster mixing proportions
        pi = pm.Dirichlet("pi", a=np.ones(K) * 2.0)
        
        # Priors: Feature distributions (Student-T for robustness)
        mu = pm.Normal("mu", mu=0.0, sigma=1.0, shape=(K, D))
        sigma = pm.HalfNormal("sigma", sigma=1.0, shape=(K, D))
        nu = pm.Gamma("nu", alpha=3.0, beta=0.3, shape=K)  # degrees of freedom
        
        # Priors: Degradation probability per cluster
        theta = pm.Beta("theta", alpha=2.0, beta=2.0, shape=K)
        
        # Log-likelihood: Features ~ Student-T
        dist_x = pm.StudentT.dist(nu=nu[:, None], mu=mu, sigma=sigma)
        logp_x = pm.logp(dist_x, X_data[:, None, :]).sum(axis=2)  # shape: (N, K)
        
        # Log-likelihood: Labels ~ Bernoulli
        dist_y = pm.Bernoulli.dist(p=theta)
        logp_y = pm.logp(dist_y, y_data[:, None])  # shape: (N, K)
        
        # Mixture log-likelihood
        logp_components = logp_x + logp_y + pt.log(pi)
        pm.Potential("mixture_loglikelihood", 
                     pt.logsumexp(logp_components, axis=1).sum())
    
    print("✓ Model built successfully")
    return model


def sample_posterior(model, config):
    """
    Sample from posterior using MCMC
    Optimized for M1 Max CPU
    """
    print(f"\n{'='*60}")
    print("MCMC Sampling Configuration")
    print(f"{'='*60}")
    print(f"Backend: CPU (M1 Max)")
    print(f"Chains: {config.N_CHAINS} parallel")
    print(f"Cores: {config.N_CORES}")
    print(f"Draws: {config.N_DRAWS} (tune: {config.N_TUNE})")
    print(f"Target accept: {config.TARGET_ACCEPT}")
    print(f"\n🚀 Starting sampling (this may take several minutes)...")
    
    with model:
        trace = pm.sample(
            draws=config.N_DRAWS,
            tune=config.N_TUNE,
            chains=config.N_CHAINS,
            cores=config.N_CORES,
            target_accept=config.TARGET_ACCEPT,
            progressbar=True,
            random_seed=config.RANDOM_SEED,
            return_inferencedata=True,
            idata_kwargs={"log_likelihood": False}
        )
    
    print("\n✓ Sampling complete!")
    return trace


def assign_clusters_from_posterior(trace, X, y, K):
    """
    Compute cluster assignments from posterior samples
    Returns mode (most frequent assignment) for each sample
    """
    N = X.shape[0]
    
    # Extract and flatten posterior samples
    pi_samples = trace.posterior["pi"].values
    mu_samples = trace.posterior["mu"].values
    sigma_samples = trace.posterior["sigma"].values
    nu_samples = trace.posterior["nu"].values
    theta_samples = trace.posterior["theta"].values
    
    n_chains, n_draws = pi_samples.shape[0], pi_samples.shape[1]
    n_samples = n_chains * n_draws
    
    pi_flat = pi_samples.reshape(n_samples, K)
    mu_flat = mu_samples.reshape(n_samples, K, X.shape[1])
    sigma_flat = sigma_samples.reshape(n_samples, K, X.shape[1])
    nu_flat = nu_samples.reshape(n_samples, K)
    theta_flat = theta_samples.reshape(n_samples, K)
    
    print(f"\n{'='*60}")
    print(f"Computing cluster assignments from {n_samples:,} posterior samples")
    print(f"{'='*60}")
    
    z_samples = np.zeros((n_samples, N), dtype=int)
    
    from scipy.stats import t as student_t
    
    for s in tqdm(range(n_samples), desc="Posterior samples"):
        logp_components = np.zeros((N, K))
        
        for k in range(K):
            # Student-T log-likelihood for features
            logp_x = student_t.logpdf(
                X, df=nu_flat[s, k], 
                loc=mu_flat[s, k], 
                scale=sigma_flat[s, k]
            ).sum(axis=1)
            
            # Bernoulli log-likelihood for labels
            p = theta_flat[s, k]
            logp_y = y * np.log(p + 1e-10) + (1 - y) * np.log(1 - p + 1e-10)
            
            # Component likelihood
            logp_components[:, k] = logp_x + logp_y + np.log(pi_flat[s, k] + 1e-10)
        
        z_samples[s, :] = np.argmax(logp_components, axis=1)
    
    # Posterior mode
    from scipy.stats import mode
    z_post = mode(z_samples, axis=0, keepdims=False)[0]
    
    # Compute certainty (how often mode agrees with samples)
    uncertainty = np.array([
        (z_samples[:, i] == z_post[i]).mean() for i in range(N)
    ])
    
    print(f"✓ Cluster assignments computed")
    print(f"  Mean certainty: {uncertainty.mean():.3f}")
    print(f"  Min certainty: {uncertainty.min():.3f}")
    
    return z_post, z_samples, uncertainty

In [7]:
# ============================================================================
# PART 4: EVALUATION & VISUALIZATION
# ============================================================================

def evaluate_clustering(y_true, z_pred):
    """Compute clustering quality metrics"""
    # Compute metrics
    ari = adjusted_rand_score(y_true, z_pred)
    nmi = normalized_mutual_info_score(y_true, z_pred)
    ami = adjusted_mutual_info_score(y_true, z_pred)
    fmi = fowlkes_mallows_score(y_true, z_pred)
    
    # Align clusters to labels
    conf_matrix = confusion_matrix(y_true, z_pred)
    cost_matrix = -conf_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    cluster_map = {col_ind[i]: row_ind[i] for i in range(len(row_ind))}
    
    aligned_pred = np.array([cluster_map.get(c, -1) for c in z_pred])
    accuracy = accuracy_score(y_true, aligned_pred)
    
    metrics = {
        'ARI': ari,
        'NMI': nmi,
        'AMI': ami,
        'FMI': fmi,
        'Accuracy': accuracy
    }
    
    return metrics, conf_matrix, cluster_map, aligned_pred


def print_detailed_results(y_true, z_pred, metrics, conf_matrix, cluster_map):
    """Print comprehensive clustering results"""
    print(f"\n{'='*60}")
    print("CLUSTERING RESULTS")
    print(f"{'='*60}")
    
    comparison = pd.DataFrame({
        'cluster': z_pred,
        'true_label': y_true
    })
    
    print("\n📊 Cluster distribution:")
    display(comparison['cluster'].value_counts().sort_index())
    
    print("\n📈 Degraded rate by cluster:")
    display(comparison.groupby('cluster')['true_label'].mean().round(3))
    
    print(f"\n{'='*60}")
    print("QUALITY METRICS")
    print(f"{'='*60}")
    for name, value in metrics.items():
        print(f"  {name:20s}: {value:.4f}")
    
    print(f"\n🔗 Cluster-to-label mapping: {cluster_map}")
    
    print(f"\n{'='*60}")
    print("CONFUSION MATRIX")
    print(f"{'='*60}")
    print(conf_matrix)
    
    print(f"\n{'='*60}")
    print("PER-CLUSTER ANALYSIS")
    print(f"{'='*60}")
    
    for cluster_id in sorted(comparison['cluster'].unique()):
        cluster_data = comparison[comparison['cluster'] == cluster_id]
        label_dist = cluster_data['true_label'].value_counts()
        
        if len(label_dist) > 0:
            dominant = label_dist.index[0]
            purity = label_dist.iloc[0] / len(cluster_data)
            
            print(f"\n🎯 Cluster {cluster_id} (n={len(cluster_data):,}):")
            print(f"   Dominant: {'Clean' if dominant==0 else 'Degraded'}")
            print(f"   Purity: {purity:.1%}")
            print(f"   Degraded rate: {cluster_data['true_label'].mean():.3f}")


def plot_comprehensive_results(y_true, z_pred, metrics, conf_matrix, uncertainty, aligned_pred):
    """Create comprehensive visualization dashboard"""
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Confusion matrix
    ax1 = fig.add_subplot(gs[0, 0])
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', ax=ax1, cbar_kws={'label': 'Count'})
    ax1.set_title('Confusion Matrix', fontsize=14, weight='bold')
    ax1.set_xlabel('Predicted Cluster')
    ax1.set_ylabel('True Label')
    
    # 2. Normalized confusion
    ax2 = fig.add_subplot(gs[0, 1])
    conf_norm = conf_matrix.astype('float') / (conf_matrix.sum(axis=1)[:, np.newaxis] + 1e-10)
    sns.heatmap(conf_norm, annot=True, fmt='.2f', cmap='Greens', ax=ax2, cbar_kws={'label': 'Proportion'})
    ax2.set_title('Normalized Confusion Matrix', fontsize=14, weight='bold')
    ax2.set_xlabel('Predicted Cluster')
    ax2.set_ylabel('True Label')
    
    # 3. Metrics
    ax3 = fig.add_subplot(gs[0, 2])
    colors = ['steelblue' if v >= 0.5 else 'coral' for v in metrics.values()]
    bars = ax3.bar(metrics.keys(), metrics.values(), color=colors)
    ax3.set_title('Clustering Quality Metrics', fontsize=14, weight='bold')
    ax3.set_ylabel('Score')
    ax3.set_ylim([0, 1])
    ax3.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Baseline')
    for i, (name, val) in enumerate(metrics.items()):
        ax3.text(i, val + 0.02, f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    ax3.legend()
    
    # 4. Distribution comparison
    ax4 = fig.add_subplot(gs[1, 0])
    dist_df = pd.DataFrame({
        'Predicted': pd.Series(z_pred).value_counts().sort_index(),
        'True': pd.Series(y_true).value_counts().sort_index()
    }).fillna(0)
    dist_df.plot(kind='bar', ax=ax4, color=['steelblue', 'coral'], width=0.8)
    ax4.set_title('Cluster/Label Distribution', fontsize=14, weight='bold')
    ax4.set_ylabel('Count')
    ax4.set_xlabel('Cluster / Label ID')
    ax4.legend(['Predicted Clusters', 'True Labels'])
    ax4.tick_params(axis='x', rotation=0)
    
    # 5. Certainty distribution
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.hist(uncertainty, bins=50, edgecolor='black', color='steelblue', alpha=0.7)
    ax5.axvline(uncertainty.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Mean: {uncertainty.mean():.3f}')
    ax5.axvline(uncertainty.median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {np.median(uncertainty):.3f}')
    ax5.set_title('Assignment Certainty Distribution', fontsize=14, weight='bold')
    ax5.set_xlabel('Posterior Probability')
    ax5.set_ylabel('Count')
    ax5.legend()
    
    # 6. Certainty vs correctness
    ax6 = fig.add_subplot(gs[1, 2])
    correct = (y_true == aligned_pred).astype(int)
    ax6.scatter(uncertainty[correct == 1], np.random.normal(1, 0.02, sum(correct == 1)),
                alpha=0.3, s=20, label='Correct', color='green')
    ax6.scatter(uncertainty[correct == 0], np.random.normal(0, 0.02, sum(correct == 0)),
                alpha=0.3, s=20, label='Incorrect', color='red')
    ax6.set_title('Certainty vs Correctness', fontsize=14, weight='bold')
    ax6.set_xlabel('Assignment Certainty')
    ax6.set_ylabel('Correct (1) / Incorrect (0)')
    ax6.set_ylim([-0.3, 1.3])
    ax6.legend()
    
    # 7. Cluster purity
    ax7 = fig.add_subplot(gs[2, 0])
    cluster_ids = sorted(np.unique(z_pred))
    purities = []
    for cid in cluster_ids:
        mask = z_pred == cid
        if mask.sum() > 0:
            labels_in_cluster = y_true[mask]
            purity = np.max(np.bincount(labels_in_cluster)) / len(labels_in_cluster)
            purities.append(purity)
        else:
            purities.append(0)
    colors_bar = ['green' if p >= 0.7 else 'orange' if p >= 0.5 else 'red' for p in purities]
    ax7.bar(cluster_ids, purities, color=colors_bar)
    ax7.set_title('Cluster Purity', fontsize=14, weight='bold')
    ax7.set_xlabel('Cluster ID')
    ax7.set_ylabel('Purity')
    ax7.set_ylim([0, 1])
    ax7.axhline(y=0.7, color='green', linestyle='--', alpha=0.3, label='Good (≥0.7)')
    ax7.axhline(y=0.5, color='orange', linestyle='--', alpha=0.3, label='Fair (≥0.5)')
    ax7.legend()
    
    # 8. Label recall
    ax8 = fig.add_subplot(gs[2, 1])
    label_ids = sorted(np.unique(y_true))
    recalls = []
    for lid in label_ids:
        mask = y_true == lid
        if mask.sum() > 0:
            clusters_for_label = z_pred[mask]
            recall = np.max(np.bincount(clusters_for_label)) / len(clusters_for_label)
            recalls.append(recall)
        else:
            recalls.append(0)
    colors_recall = ['green' if r >= 0.7 else 'orange' if r >= 0.5 else 'red' for r in recalls]
    ax8.bar(label_ids, recalls, color=colors_recall)
    ax8.set_title('Label Recall', fontsize=14, weight='bold')
    ax8.set_xlabel('True Label')
    ax8.set_ylabel('Recall')
    ax8.set_ylim([0, 1])
    ax8.axhline(y=0.7, color='green', linestyle='--', alpha=0.3)
    ax8.axhline(y=0.5, color='orange', linestyle='--', alpha=0.3)
    
    # 9. Summary statistics
    ax9 = fig.add_subplot(gs[2, 2])
    ax9.axis('off')
    summary_text = f"""
    SUMMARY STATISTICS
    {'='*35}
    
    Total Samples: {len(y_true):,}
    Number of Clusters: {len(np.unique(z_pred))}
    
    Certainty:
      Mean: {uncertainty.mean():.3f}
      Median: {np.median(uncertainty):.3f}
      Min: {uncertainty.min():.3f}
      Max: {uncertainty.max():.3f}
    
    Agreement with True Labels:
      Overall: {(y_true == aligned_pred).mean():.1%}
      Clean samples: {(y_true[y_true==0] == aligned_pred[y_true==0]).mean():.1%}
      Degraded samples: {(y_true[y_true==1] == aligned_pred[y_true==1]).mean():.1%}
    """
    ax9.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
             verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.suptitle('Bayesian Clustering Analysis Dashboard', fontsize=16, weight='bold', y=0.995)
    return fig




In [8]:
# ============================================================================
# PART 5: MAIN PIPELINE
# ============================================================================

def run_complete_pipeline(config, skip_feature_extraction=False, features_path=None):
    """
    Execute complete Bayesian clustering pipeline
    
    Parameters:
    -----------
    config : Config object
    skip_feature_extraction : bool
        If True, load features from features_path instead of extracting
    features_path : str
        Path to saved features CSV (if skip_feature_extraction=True)
    
    Returns:
    --------
    results : dict with all outputs
    """
    results = {}
    
    # STEP 1: Load database and create labels
    print(f"\n{'#'*60}")
    print("# STEP 1: LOAD DATA & CREATE LABELS")
    print(f"{'#'*60}")
    
    db = load_ptbxl_database(config.PTBXL_ROOT)
    y = create_degradation_labels(db)
    results['db'] = db
    results['y_true'] = y
    
    # STEP 2: Feature extraction or loading
    print(f"\n{'#'*60}")
    print("# STEP 2: FEATURE EXTRACTION")
    print(f"{'#'*60}")
    
    if skip_feature_extraction and features_path and os.path.exists(features_path):
        print(f"Loading pre-extracted features from: {features_path}")
        feat_df = pd.read_csv(features_path)
        print(f"✓ Loaded features: shape {feat_df.shape}")
    else:
        feat_df = build_feature_dataframe(
            db, y, config.PTBXL_ROOT, 
            n_records=config.MAX_RECORDS, 
            use_hr=True
        )
        # Save features
        feat_df.to_csv("ecg_features.csv", index=False)
        print("✓ Features saved to 'ecg_features.csv'")
    
    results['features'] = feat_df
    
    # STEP 3: Prepare features
    print(f"\n{'#'*60}")
    print("# STEP 3: PREPARE FEATURES FOR CLUSTERING")
    print(f"{'#'*60}")
    
    X, y_prep, X_mean, X_std = prepare_features_for_clustering(feat_df)
    results['X'] = X
    results['y_prep'] = y_prep
    results['X_mean'] = X_mean
    results['X_std'] = X_std
    
    # STEP 4: Build model
    print(f"\n{'#'*60}")
    print("# STEP 4: BUILD BAYESIAN MODEL")
    print(f"{'#'*60}")
    
    model = build_bayesian_model(X, y_prep, config.N_CLUSTERS)
    results['model'] = model
    
    # STEP 5: Sample posterior
    print(f"\n{'#'*60}")
    print("# STEP 5: SAMPLE POSTERIOR")
    print(f"{'#'*60}")
    
    trace = sample_posterior(model, config)
    results['trace'] = trace
    
    # Save trace
    trace.to_netcdf("bayesian_trace.nc")
    print("✓ Trace saved to 'bayesian_trace.nc'")
    
    # STEP 6: Assign clusters
    print(f"\n{'#'*60}")
    print("# STEP 6: COMPUTE CLUSTER ASSIGNMENTS")
    print(f"{'#'*60}")
    
    z_post, z_samples, uncertainty = assign_clusters_from_posterior(
        trace, X, y_prep, config.N_CLUSTERS
    )
    results['z_pred'] = z_post
    results['z_samples'] = z_samples
    results['uncertainty'] = uncertainty
    
    # STEP 7: Evaluate
    print(f"\n{'#'*60}")
    print("# STEP 7: EVALUATE CLUSTERING")
    print(f"{'#'*60}")
    
    metrics, conf_matrix, cluster_map, aligned_pred = evaluate_clustering(
        feat_df['y'].values, z_post
    )
    results['metrics'] = metrics
    results['conf_matrix'] = conf_matrix
    results['cluster_map'] = cluster_map
    results['aligned_pred'] = aligned_pred
    
    # STEP 8: Print results
    print_detailed_results(
        feat_df['y'].values, z_post, metrics, conf_matrix, cluster_map
    )
    
    # STEP 9: Visualize
    print(f"\n{'#'*60}")
    print("# STEP 8: VISUALIZE RESULTS")
    print(f"{'#'*60}")
    
    fig = plot_comprehensive_results(
        feat_df['y'].values, z_post, metrics, conf_matrix, 
        uncertainty, aligned_pred
    )
    plt.show()
    results['fig'] = fig
    
    # STEP 10: Save results
    print(f"\n{'#'*60}")
    print("# STEP 9: SAVE RESULTS")
    print(f"{'#'*60}")
    
    results_df = pd.DataFrame({
        'ecg_id': feat_df['ecg_id'].values,
        'cluster': z_post,
        'true_label': feat_df['y'].values,
        'certainty': uncertainty,
        'aligned_cluster': aligned_pred
    })
    results_df.to_csv('bayesian_clustering_results.csv', index=False)
    print("✓ Results saved to 'bayesian_clustering_results.csv'")
    
    results['results_df'] = results_df
    
    print(f"\n{'='*60}")
    print("✓ PIPELINE COMPLETE!")
    print(f"{'='*60}")
    
    return results


In [ ]:
# ============================================================================
# USAGE EXAMPLES
# ============================================================================


# 1. Quick start (process subset for testing)
config = Config()
config.MAX_RECORDS = 500  # Start small for testing
results = run_complete_pipeline(config)

# 2. Full dataset
config = Config()
config.MAX_RECORDS = None  # Process all records
results = run_complete_pipeline(config)

# 3. Re-run clustering on existing features (skip extraction)
results = run_complete_pipeline(
    config, 
    skip_feature_extraction=True, 
    features_path="ecg_features.csv"
)

# 4. Access results
clusters = results['z_pred']
metrics = results['metrics']
trace = results['trace']
uncertainty = results['uncertainty']

# 5. Diagnostic plots
az.plot_trace(trace, var_names=['pi', 'theta'])
plt.tight_layout()
plt.show()

az.plot_posterior(trace, var_names=['pi', 'theta'])
plt.show()




############################################################
# STEP 1: LOAD DATA & CREATE LABELS
############################################################
✓ Loaded 21,837 records from PTB-XL database
✓ Created labels: 16,816 clean, 5,021 degraded
  Degradation rate: 23.0%

############################################################
# STEP 2: FEATURE EXTRACTION
############################################################

Extracting features from 500 records...


Processing ECGs: 100%|███████████████████████| 500/500 [00:00<00:00, 504.41it/s]



✓ Extracted features: shape (500, 18)
  Features: fs, n_samples, n_channels, x_mean, x_std, x_rms, x_skew, x_kurtosis, p_base, p_qrs, p_hf, p_tot, psqi, bassqi, hf_ratio
✓ Features saved to 'ecg_features.csv'

############################################################
# STEP 3: PREPARE FEATURES FOR CLUSTERING
############################################################

Data prepared for clustering
Samples: 500
Features: 15
Clean: 344 | Degraded: 156

############################################################
# STEP 4: BUILD BAYESIAN MODEL
############################################################

Building Bayesian Mixture Model
N = 500 samples
D = 15 features
K = 4 clusters
✓ Model built successfully

############################################################
# STEP 5: SAMPLE POSTERIOR
############################################################

MCMC Sampling Configuration
Backend: CPU (M1 Max)
Chains: 4 parallel
Cores: 8
Draws: 2000 (tune: 1000)
Target accept: 0.95

🚀 Star

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [pi, mu, sigma, nu, theta]


In [ ]:
len(db)

In [ ]:
X.shape

In [ ]:
import sys
print(sys.executable)


In [ ]:
# import sys
# print(sys.version)
# print(sys.executable)
# import sys
# !{sys.executable} -m pip install -U pip
# !{sys.executable} -m pip install -U numpy wheel
# !{sys.executable} -m pip install -U jax-metal
# import jax
# print("JAX version:", jax.__version__)
# print("Backend:", jax.default_backend())
# print("Devices:", jax.devices())




In [ ]:
Xz.shape

In [ ]:
# 6. Save/load trace for later analysis
trace.to_netcdf("my_trace.nc")
trace = az.from_netcdf("my_trace.nc")

In [ ]:
import numpy as np
import time
import platform

def cpu_matrix_multiply(a, b):
    """Standard CPU matrix multiplication"""
    return np.dot(a, b)

def benchmark_computation(size=2000, iterations=5):
    """Benchmark matrix operations on CPU"""
    print(f"System: {platform.system()} {platform.machine()}")
    print(f"Python numpy using: {np.__config__.show()}\n")
    print(f"Matrix size: {size}x{size}")
    print(f"Iterations: {iterations}\n")
    
    # Generate random matrices
    print("Generating random matrices...")
    a = np.random.randn(size, size).astype(np.float32)
    b = np.random.randn(size, size).astype(np.float32)
    
    # Warm-up
    print("Warming up...")
    _ = cpu_matrix_multiply(a, b)
    
    # CPU Benchmark
    print("\n--- CPU Benchmark ---")
    cpu_times = []
    for i in range(iterations):
        start = time.time()
        result_cpu = cpu_matrix_multiply(a, b)
        cpu_time = time.time() - start
        cpu_times.append(cpu_time)
        print(f"Iteration {i+1}: {cpu_time:.4f} seconds")
    
    avg_cpu = np.mean(cpu_times)
    print(f"\nAverage CPU time: {avg_cpu:.4f} seconds")
    print(f"Operations: {2 * size**3 / 1e9:.2f} billion")
    print(f"Performance: {2 * size**3 / avg_cpu / 1e9:.2f} GFLOPS")

if __name__ == "__main__":
    print("=" * 60)
    print("M1 Mac Computation Benchmark")
    print("=" * 60)
    print("\nNote: NumPy on M1 Macs automatically uses the Accelerate")
    print("framework which leverages the GPU/Neural Engine for")
    print("certain operations like matrix multiplication.\n")
    
    benchmark_computation(size=2000, iterations=3)
    
    print("\n" + "=" * 60)
    print("Try different sizes: benchmark_computation(size=3000)")
    print("=" * 60)

In [ ]:
# Install PyTorch with MPS support
pip install torch torchvision

# Then run this test:
python -c "
import torch
import time

# Check if MPS is available
print(f'MPS available: {torch.backends.mps.is_available()}')

# Create tensors on GPU
size = 2000
a = torch.randn(size, size, device='mps')
b = torch.randn(size, size, device='mps')

# Benchmark
start = time.time()
c = torch.matmul(a, b)
torch.mps.synchronize()  # Wait for GPU to finish
print(f'GPU time: {time.time() - start:.4f} seconds')
"